# LocalLLM Benchmark: Quantisation Optimisation & Inference Analysis
### Task 3 - tndrr AI Internship Technical Evaluation

**Author:** Senior Machine Learning Engineer & Performance Benchmarking Researcher  
**Model:** `Qwen/Qwen2.5-1.5B-Instruct` (GGUF Architecture)  
**Evaluated Quantisations:** `Q4_K_M` (4-bit Medium K-quant) vs `Q8_0` (8-bit Integer Quant)  
**Execution Engine:** llama.cpp (Official GGML Release Build b12532)  
**Host Machine:** HP Victus 15-fb0xxx | AMD Ryzen 5 5600H | AMD Radeon RX 6500M | 16 GB RAM

## 1. Hardware Environment & Runtime Discovery
Inspect the physical host topology, available memory, CPU vector capabilities, and graphics acceleration.

In [ ]:
import json, os
import pandas as pd
import matplotlib.pyplot as plt

hw_profile_path = os.path.join('..', 'results', 'hardware_profile.json')
with open(hw_profile_path, 'r') as f:
    hw = json.load(f)

print(f"OS: {hw['platform']['os']} {hw['platform']['os_release']}")
print(f"Processor: {hw['cpu']['processor']} ({hw['cpu']['physical_cores']} Cores / {hw['cpu']['logical_cores']} Threads)")
print(f"RAM: {hw['memory']['total_ram_gb']} GB Total (Available: {hw['memory']['available_ram_gb']} GB)")
print(f"Storage D: Free {hw['storage'].get('D:\\', {}).get('free_gb')} GB")
print(f"Vulkan 1.3 Acceleration: {hw['gpu']['vulkan_installed']} ({len(hw['gpu']['devices'])} devices detected)")

## 2. Model Architecture & GGUF Parity Verification
Verify that both `Q4_K_M` and `Q8_0` configurations share the identical upstream transformer architecture and layer parameters.

In [ ]:
import sys
sys.path.append(os.path.abspath('../src'))
from model_manager import ModelManager

mgr = ModelManager()
model_verification = mgr.verify_models()
for q_name in ['Q4_K_M', 'Q8_0']:
    info = model_verification.get(q_name, {})
    print(f"[{q_name}] File Size: {info.get('file_size_gb')} GB | Arch: {info.get('architecture')} | Layers: {info.get('block_count')} | Heads: {info.get('head_count')} (KV: {info.get('head_count_kv')})")

print("Architectural Parity:", model_verification.get('architectural_parity'))

## 3. Benchmark Results & Comparative Analysis
Load empirical measurements across all 10 evaluation prompts and compare generation tokens per second, prompt processing throughput, and memory consumption.

In [ ]:
results_json_path = os.path.join('..', 'results', 'benchmark_results.json')
with open(results_json_path, 'r') as f:
    bench_data = json.load(f)

summary_df = pd.DataFrame(bench_data['summary']).T
display_cols = ['quant_type', 'model_file_size_gb', 'generation_tps_mean', 'prompt_eval_tps_mean', 'latency_sec_mean', 'peak_ram_gb_max', 'average_quality_score']
summary_df[display_cols]

## 4. Visualizations: Throughput, Memory & Output Quality

In [ ]:
from IPython.display import Image, display
for chart_name in ['performance_comparison.png', 'memory_comparison.png', 'quality_comparison.png']:
    path = os.path.join('..', 'results', chart_name)
    if os.path.exists(path):
        print(f"Displaying: {chart_name}")
        display(Image(filename=path))
    else:
        print(f"Chart not found: {path}")

## 5. Output Quality Evaluation Across 10 Capabilities
Inspect the scored comparisons across all 10 domain prompts.

In [ ]:
quality_csv_path = os.path.join('..', 'results', 'quality_comparison.csv')
quality_df = pd.read_csv(quality_csv_path)
quality_df[['prompt_id', 'category', 'q4_score', 'q8_score', 'difference_explanation']]

## 6. KV Cache & 20-User GPU Serving Analytical Model
Calculate the memory requirements for serving 20 concurrent users on a single 24 GB GPU.

In [ ]:
def calculate_kv_cache_mb(layers, kv_heads, head_dim, seq_len, precision_bytes, batch_size):
    total_bytes = 2 * layers * kv_heads * head_dim * seq_len * precision_bytes * batch_size
    return total_bytes / (1024**2)

# Qwen2.5-7B with 20 concurrent sequences at 4096 context
qwen7b_kv_fp16 = calculate_kv_cache_mb(layers=28, kv_heads=4, head_dim=128, seq_len=4096, precision_bytes=2.0, batch_size=20)
qwen7b_kv_fp8 = calculate_kv_cache_mb(layers=28, kv_heads=4, head_dim=128, seq_len=4096, precision_bytes=1.0, batch_size=20)

print(f"Qwen2.5-7B (20 users @ 4096 context):")
print(f"  - FP16 KV Cache: {qwen7b_kv_fp16:.2f} MB ({qwen7b_kv_fp16/1024:.2f} GB)")
print(f"  - FP8 KV Cache:  {qwen7b_kv_fp8:.2f} MB ({qwen7b_kv_fp8/1024:.2f} GB)")

# Total VRAM Budget on 24 GB
weights_mb = 4966.0  # 4-bit AWQ
runtime_mb = 2200.0
safety_mb = 3000.0
total_used_mb = weights_mb + qwen7b_kv_fp16 + runtime_mb + safety_mb
print(f"\nTotal VRAM Allocated: {total_used_mb:.2f} MB / 24,576 MB ({total_used_mb/24576*100:.1f}%)")
print(f"Unallocated PagedAttention Buffer: {24576 - total_used_mb:.2f} MB")